# Tree-Based Methods (as used in the course)

**Course:** BUSI70575 — Systematic Trading Strategies with ML
**Sources:** Sessions 4 (regression), 5 (classification), 6 — all via `scikit-learn` / `xgboost`.

The course never builds a tree from scratch: it uses **Random Forest** and **XGBoost** as the
meta-model. This notebook explains *what those estimators compute* — how a split is scored, how a
forest aggregates, how boosting stacks weak learners — and reproduces the key numbers against the
libraries. Depth is deliberately **course-code-anchored** (the split criterion the course sets);
AdaBoost / Gini / from-scratch CART derivations are out of scope by choice.

Run top-to-bottom with the **`stml`** kernel.

---

## 1. The two estimators

| | Random Forest | XGBoost |
|---|---|---|
| ensemble style | **bagging** (parallel) | **boosting** (sequential) |
| what it reduces | variance | bias |
| trees | deep, decorrelated | shallow, corrective |
| course use | Session 5 classifier, Session 4 regressor | Sessions 4 & 5 |

Everything below uses one synthetic binary dataset so the numbers are reproducible offline.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
import xgboost as xgb

np.random.seed(42)

## 2. How a tree decides a split

A CART tree grows greedily: at each node it searches every feature and threshold and keeps the split
that most reduces an **impurity** measure. The course sets `criterion='entropy'`, so the numbers are:

**Entropy of a node** (Shannon, base 2 — this is what `sklearn` uses internally):
$$H = -\sum_{k} p_k \log_2 p_k,\qquad p_k=\text{fraction of class }k.$$

**Information gain of a split** (parent entropy minus the sample-weighted entropy of the children):
$$\mathrm{IG} = H(\text{parent}) - \sum_{c\in\{L,R\}} \frac{n_c}{n}\,H(c).$$

The chosen split maximises IG (equivalently, minimises the weighted child entropy).

> sklearn's **default** is Gini, $G=1-\sum_k p_k^2$; **regression** trees use MSE. The course uses
> entropy, so we work with entropy — the splitting *procedure* is identical for any impurity.

### Worked example (by hand)

A node with **8 samples, 4 positive / 4 negative**, split into two children of 4:

$$H(\text{parent}) = -\tfrac12\log_2\tfrac12 -\tfrac12\log_2\tfrac12 = 1.0\ \text{bit}.$$

Child L $=(3\text{ pos},1\text{ neg})$, Child R $=(1\text{ pos},3\text{ neg})$:
$$H = -\tfrac34\log_2\tfrac34 - \tfrac14\log_2\tfrac14 = 0.311 + 0.5 = 0.8113\ \text{(each)}.$$

Weighted child entropy $= \tfrac48(0.8113)+\tfrac48(0.8113)=0.8113$, so
$$\mathrm{IG} = 1.0 - 0.8113 = \mathbf{0.1887}.$$

In [ ]:
def entropy(counts):
    """Shannon entropy (base 2) of class counts (or proportions -- scale invariant)."""
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

H_parent = entropy([4, 4])
H_child  = entropy([3, 1])
IG = H_parent - (4 / 8) * entropy([3, 1]) - (4 / 8) * entropy([1, 3])
print(f"H(parent) [4,4] = {H_parent:.4f}")
print(f"H(child)  [3,1] = {H_child:.4f}")
print(f"information gain = {IG:.4f}")

assert np.isclose(H_parent, 1.0)
assert np.isclose(H_child, 0.8113, atol=1e-4)
assert np.isclose(IG, 0.1887, atol=1e-4)
print("OK: matches the hand-worked numbers")

In [ ]:
# Confirm sklearn's 'entropy' criterion really is base-2 entropy, on a real fitted split.
Xc, yc = make_classification(n_samples=300, n_features=6, n_informative=4,
                             n_redundant=0, random_state=1)
tree = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=0).fit(Xc, yc)
t = tree.tree_

l, r = t.children_left[0], t.children_right[0]
wP, wL, wR = t.weighted_n_node_samples[[0, l, r]]
ig_tree   = t.impurity[0] - (wL * t.impurity[l] + wR * t.impurity[r]) / wP
ig_manual = entropy(t.value[0][0]) - (wL * entropy(t.value[l][0])
                                      + wR * entropy(t.value[r][0])) / wP

print(f"root entropy : sklearn={t.impurity[0]:.4f}   formula={entropy(t.value[0][0]):.4f}")
print(f"info gain    : sklearn={ig_tree:.4f}   formula={ig_manual:.4f}")
assert np.isclose(t.impurity[0], entropy(t.value[0][0]))   # base-2 entropy
assert np.isclose(ig_tree, ig_manual)
print("OK: sklearn's entropy split == the hand formula")

## 3. Random Forest = bagging + feature subsampling

Two sources of randomness decorrelate the trees so their *average* has lower variance than any one tree:

1. **Bagging (bootstrap aggregating).** Each tree trains on a bootstrap sample (draw $n$ rows *with
   replacement*). A row is left out of a given tree with probability
   $\left(1-\tfrac1n\right)^n \to e^{-1}\approx 0.368$. Those **out-of-bag (OOB)** rows give a free
   validation estimate (`oob_score_`).
2. **Feature subsampling.** At each split only `max_features` features are candidates
   (`'sqrt'` $=\sqrt{p}$). This stops one strong feature from dominating every tree.

**Prediction.** Classification averages the trees' class probabilities (then argmax); regression averages
their outputs. The exact Session 5 configuration is used below.

In [ ]:
print("(1 - 1/n)^n  ->  1/e ?")
for n in [10, 100, 1000, 100000]:
    print(f"  n={n:>6}:  {(1 - 1/n)**n:.4f}")
print(f"  1/e      =  {1/np.e:.4f}")
assert np.isclose((1 - 1/100000)**100000, 1/np.e, atol=1e-3)

rng = np.random.default_rng(0)
n = 2000
boot = rng.integers(0, n, size=n)                  # one bootstrap sample of row indices
oob_frac = 1 - len(np.unique(boot)) / n
print(f"\nempirical OOB fraction (n={n}): {oob_frac:.4f}   (theory ~ {1/np.e:.4f})")
assert abs(oob_frac - 1/np.e) < 0.03

In [ ]:
X, y = make_classification(n_samples=800, n_features=10, n_informative=5,
                           n_redundant=2, random_state=42)
X = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
y = pd.Series(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# --- Verbatim Session 5 config (cell 39); oob_score=True added for the OOB demo ---
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_split=5, min_samples_leaf=2,
    max_features='sqrt', criterion='entropy', random_state=42, n_jobs=-1,
    oob_score=True,
)
rf_model.fit(X_train, y_train)
print("OOB score (held-out estimate):", round(rf_model.oob_score_, 4))

# A forest's class probabilities are the AVERAGE of its trees' probabilities:
proba_forest = rf_model.predict_proba(X_test)
proba_trees  = np.mean([est.predict_proba(X_test) for est in rf_model.estimators_], axis=0)
assert np.allclose(proba_forest, proba_trees)
print("OK: forest predict_proba == mean of the 100 trees' predict_proba")

## 4. XGBoost = gradient boosting

Boosting builds trees **sequentially**, each correcting the current model's errors. After $m$ rounds the
additive model is
$$F_m(x) = F_{m-1}(x) + \eta\, h_m(x),$$
where $h_m$ is a new tree fit to the negative gradient (pseudo-residuals) of the loss and $\eta=$
`learning_rate` shrinks each step. Key knobs (Session 5 values):

* `n_estimators` — number of boosting rounds (trees added in sequence).
* `learning_rate` ($\eta=0.05$) — shrinkage; smaller needs more rounds but generalises better.
* `max_depth` (4) — depth of each booster; boosters are *shallow* (weak learners).
* `subsample` (0.8), `colsample_bytree` (0.8) — row / column sampling per tree (stochastic boosting).
* `reg_alpha` (L1) / `reg_lambda` (L2) — penalise leaf weights to curb overfitting.
* `objective='binary:logistic'`, `eval_metric='logloss'` — loss optimised and tracked.

Bagging is parallel (variance reduction); boosting is sequential (**bias** reduction by stacking weak learners).

In [ ]:
# --- Verbatim Session 5 config (cell 43) ---
xgb_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    objective='binary:logistic', eval_metric='logloss', random_state=42,
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_train, y_train), (X_test, y_test)], verbose=False)

res = xgb_model.evals_result()
train_ll, test_ll = res['validation_0']['logloss'], res['validation_1']['logloss']
print(f"train logloss: {train_ll[0]:.4f} -> {train_ll[-1]:.4f}")
print(f"test  logloss: {test_ll[0]:.4f} -> {test_ll[-1]:.4f}")
assert train_ll[-1] < train_ll[0]                  # boosting drives the loss down

plt.figure(figsize=(7, 4))
plt.plot(train_ll, label="train")
plt.plot(test_ll, label="test")
plt.xlabel("boosting round"); plt.ylabel("logloss")
plt.title("XGBoost: sequential error reduction (round by round)")
plt.legend(); plt.tight_layout(); plt.show()

## 5. Reading the outputs — `evaluate_model`

The course's `evaluate_model` reports accuracy / precision / recall / F1 / AUC on **train and test**.
The train-vs-test gap is the overfitting signal; AUC uses `predict_proba` (ranking quality), the others
use the hard `predict` label. Trees overfit easily, so expect train metrics above test.

In [ ]:
# --- Verbatim from Solution_Programming_Session_5.ipynb (cell 72) ---
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Comprehensive model evaluation"""
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_proba = model.predict_proba(X_train)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'Model': model_name,
        'Train Accuracy': accuracy_score(y_train, train_pred),
        'Test Accuracy': accuracy_score(y_test, test_pred),
        'Train Precision': precision_score(y_train, train_pred),
        'Test Precision': precision_score(y_test, test_pred),
        'Train Recall': recall_score(y_train, train_pred),
        'Test Recall': recall_score(y_test, test_pred),
        'Train F1': f1_score(y_train, train_pred),
        'Test F1': f1_score(y_test, test_pred),
        'Train AUC': roc_auc_score(y_train, train_proba),
        'Test AUC': roc_auc_score(y_test, test_proba),
    }
    metrics['test_proba'] = test_proba
    return metrics

In [ ]:
rows = [
    evaluate_model(rf_model,  X_train, y_train, X_test, y_test, "RandomForest"),
    evaluate_model(xgb_model, X_train, y_train, X_test, y_test, "XGBoost"),
]
comp = pd.DataFrame(rows).drop(columns=["test_proba"]).set_index("Model")
print(comp.round(3).to_string())
comp.round(3)

## 6. Hyperparameter cheat-sheet (Sessions 4–6)

**Random Forest**

| param | meaning | effect |
|---|---|---|
| `n_estimators` | number of trees | more = lower variance, slower; plateaus |
| `max_depth` | max tree depth | deeper = more variance / overfit |
| `min_samples_split` | min samples to split a node | larger = simpler trees |
| `min_samples_leaf` | min samples per leaf | larger = smoother, less overfit |
| `max_features` | candidates per split (`'sqrt'`) | smaller = more decorrelated trees |
| `criterion` | split impurity (`'entropy'`) | entropy / gini for classification |
| `bootstrap` | sample rows with replacement | enables bagging + OOB |
| `oob_score` | score on out-of-bag rows | free validation estimate |
| `class_weight` | reweight classes | handle class imbalance |

**XGBoost**

| param | meaning | effect |
|---|---|---|
| `n_estimators` | boosting rounds | more = lower bias, risk overfit |
| `learning_rate` ($\eta$) | step shrinkage | smaller = needs more rounds, generalises better |
| `max_depth` | booster depth | controls interaction order / complexity |
| `subsample` | row fraction per tree | $<1$ adds regularisation (stochastic) |
| `colsample_bytree` | feature fraction per tree | $<1$ decorrelates boosters |
| `reg_alpha` / `reg_lambda` | L1 / L2 on leaf weights | larger = stronger shrinkage |
| `objective` / `eval_metric` | loss / monitored metric | task-specific |

**Tuning (Session 4):** `RandomizedSearchCV` over this grid with `TimeSeriesSplit` CV and
`scoring='neg_mean_absolute_error'` — time-aware folds avoid look-ahead during model selection.

## Source pointers

| What | File | Cell |
|---|---|---|
| `RandomForestClassifier` config (entropy, sqrt) | `Solution_Programming_Session_5.ipynb` | 39 |
| `XGBClassifier` config | `Solution_Programming_Session_5.ipynb` | 43 |
| `evaluate_model` | `Solution_Programming_Session_5.ipynb` | 72 |
| RF / XGB **regression** + `featurize` | `Solution_Programming_Session_4.ipynb` | 13, 19, 26 |
| `RandomizedSearchCV` + `TimeSeriesSplit` grid | `Solution_Programming_Session_4.ipynb` | 24 |
| Feature-importance bars | `Solution_Programming_Session_4/5/6.ipynb` | — |

Trees are used only via `scikit-learn` / `xgboost` in the course; split mechanics here come from the
`criterion` the course sets. AdaBoost / from-scratch CART / Gini derivations are out of scope by design.